# Air Quality in Dhaka, Bangladesh: A Monthly Analysis (2017–2025)

This notebook presents a publication-oriented analysis of monthly PM2.5 and AQI data
for Dhaka, Bangladesh covering January 2017 through December 2025. It is designed to
serve as a reproducible companion to the *Results* section of an academic paper.

**Data sources:** US Embassy Dhaka air-quality monitor (hourly PM2.5), aggregated to
monthly statistics and merged with World Bank demographic indicators.

---
## 1. Setup & Data Loading

In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

FIGSIZE_WIDE = (12, 5)
FIGSIZE_SQUARE = (8, 6)
DPI = 150

# Paths ------------------------------------------------------------------
DATA_PATH = pathlib.Path("../data/processed/final_dhaka_aqi_dataset.csv")
ANNUAL_PATH = pathlib.Path("../outputs/annual_summary.csv")
SEASON_PATH = pathlib.Path("../outputs/seasonality_summary.csv")

# Load data ----------------------------------------------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["month_start"])
annual = pd.read_csv(ANNUAL_PATH)
seasonal = pd.read_csv(SEASON_PATH)

# Ordered season category for consistent plotting
SEASON_ORDER = ["Winter", "Pre-monsoon", "Monsoon", "Post-monsoon"]
df["season"] = pd.Categorical(df["season"], categories=SEASON_ORDER, ordered=True)
seasonal["season"] = pd.Categorical(seasonal["season"], categories=SEASON_ORDER, ordered=True)
seasonal = seasonal.sort_values("season")

print(f"Dataset shape : {df.shape}")
print(f"Period        : {df['month_start'].min():%Y-%m} to {df['month_start'].max():%Y-%m}")
print(f"Columns       : {list(df.columns)}")

The dataset comprises **108 monthly observations** spanning nine years (2017–2025).
Each row contains PM2.5 concentration statistics (mean, median, min, max),
the corresponding US EPA AQI values, data-coverage metrics, demographic indicators
(population, urbanisation, HDI, poverty rate), and normalised meteorological proxies
(wind speed and rainfall).

---
## 2. Descriptive Statistics

In [ ]:
# Exclude non-numeric and administrative/metadata columns from summary statistics
_exclude = ["year", "month", "hourly_observations", "expected_hours",
            "coverage_pct", "is_partial_month"]
desc_df = df.select_dtypes(include="number").drop(columns=_exclude, errors="ignore")
desc = desc_df.describe().T
desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)
desc.style.format(precision=2).set_caption("Table 1 — Descriptive statistics of key variables (n = 108 months)")

**Key observations from the descriptive statistics:**

- Monthly mean PM2.5 concentrations range from roughly 20 µg/m³ during the cleanest
  monsoon months to over 250 µg/m³ in peak winter episodes — far exceeding the
  WHO annual guideline of 5 µg/m³ and the Bangladesh national standard of 15 µg/m³ (annual).
- The corresponding AQI values regularly classify Dhaka's air as *Unhealthy* to
  *Very Unhealthy* (AQI > 150) for a substantial portion of the year.
- Data coverage is generally high (median ≈ 100 %), lending confidence to the
  monthly aggregates.
- Population grew steadily over the period, while HDI and poverty indicators reflect
  gradual socioeconomic development.

---
## 3. Annual AQI and PM2.5 Summaries

In [ ]:
# Display the pre-computed annual summary table
annual_display = annual.copy()
annual_display.columns = [
    "Year", "AQI (mean)", "AQI (max)", "PM2.5 (mean)",
    "PM2.5 (max)", "Observations", "Months", "Partial months",
    "Population", "HDI", "Poverty %",
]
annual_display.style.format({
    "AQI (mean)": "{:.1f}",
    "AQI (max)": "{:.1f}",
    "PM2.5 (mean)": "{:.1f}",
    "PM2.5 (max)": "{:.1f}",
    "Observations": "{:,.0f}",
    "Population": "{:,.0f}",
    "HDI": "{:.3f}",
    "Poverty %": "{:.1f}",
}).set_caption("Table 2 — Annual air-quality and socioeconomic summary for Dhaka")

In [ ]:
fig, ax1 = plt.subplots(figsize=FIGSIZE_WIDE, dpi=DPI)

x = np.arange(len(annual))
width = 0.38

bars1 = ax1.bar(x - width / 2, annual["aqi_mean_annual"], width,
                label="Mean AQI", color="#4878CF", edgecolor="white", linewidth=0.6)
ax1.set_ylabel("AQI (US EPA)", fontsize=12)
ax1.set_xlabel("Year", fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(annual["year"].astype(int))

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width / 2, annual["pm25_mean_annual"], width,
                label="Mean PM2.5 (µg/m³)", color="#D65F5F", edgecolor="white", linewidth=0.6)
ax2.set_ylabel("PM2.5 (µg/m³)", fontsize=12)

# Combine legends
handles = [bars1, bars2]
labels = [h.get_label() for h in handles]
ax1.legend(handles, labels, loc="upper left", frameon=True, fontsize=10)

ax1.set_title("Figure 1 — Annual Mean AQI and PM2.5 in Dhaka (2017–2025)", fontsize=13, pad=12)
fig.tight_layout()
plt.show()

**Interpretation (Annual trends):**

Annual mean AQI and PM2.5 concentrations exhibit a broadly **stable-to-declining
trajectory** over the study period, though absolute levels remain in the *Unhealthy*
range (AQI > 150) for every year on record. Any apparent improvement should be
interpreted cautiously, as year-to-year variation in meteorological conditions
(particularly monsoon intensity) can mask or amplify underlying emission trends.
The slight reduction in peak concentrations in later years may partially reflect
brick-kiln regulations and the shift toward natural-gas-fired kilns enacted by
the Department of Environment.

---
## 4. Seasonal Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=DPI)

# AQI boxplot
sns.boxplot(data=df, x="season", y="aqi_mean", order=SEASON_ORDER,
            palette="coolwarm", ax=axes[0], linewidth=0.8)
axes[0].set_title("(a) Monthly Mean AQI by Season", fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylabel("AQI (US EPA)", fontsize=11)

# PM2.5 boxplot
sns.boxplot(data=df, x="season", y="pm25_mean", order=SEASON_ORDER,
            palette="coolwarm", ax=axes[1], linewidth=0.8)
axes[1].set_title("(b) Monthly Mean PM2.5 by Season", fontsize=12)
axes[1].set_xlabel("")
axes[1].set_ylabel("PM2.5 (µg/m³)", fontsize=11)

fig.suptitle("Figure 2 — Seasonal Distribution of Air-Quality Indicators",
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Seasonal mean bar chart from pre-computed summary
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=DPI, sharey=False)

colours = ["#D65F5F", "#F0A830", "#4878CF", "#6ACC65"]

axes[0].bar(seasonal["season"].astype(str), seasonal["aqi_mean"], color=colours, edgecolor="white")
axes[0].set_ylabel("Mean AQI", fontsize=11)
axes[0].set_title("(a) Seasonal Mean AQI", fontsize=12)
for i, v in enumerate(seasonal["aqi_mean"]):
    axes[0].text(i, v + 2, f"{v:.1f}", ha="center", fontsize=9)

axes[1].bar(seasonal["season"].astype(str), seasonal["pm25_mean"], color=colours, edgecolor="white")
axes[1].set_ylabel("Mean PM2.5 (µg/m³)", fontsize=11)
axes[1].set_title("(b) Seasonal Mean PM2.5", fontsize=12)
for i, v in enumerate(seasonal["pm25_mean"]):
    axes[1].text(i, v + 1.5, f"{v:.1f}", ha="center", fontsize=9)

fig.suptitle("Figure 3 — Seasonal Mean Air-Quality Levels", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Seasonality summary table
seasonal_display = seasonal.copy()
seasonal_display.columns = ["Season", "AQI (mean)", "AQI (median)",
                             "PM2.5 (mean)", "PM2.5 (median)", "Months"]
seasonal_display.style.format({
    "AQI (mean)": "{:.1f}", "AQI (median)": "{:.1f}",
    "PM2.5 (mean)": "{:.1f}", "PM2.5 (median)": "{:.1f}",
}).set_caption("Table 3 — Seasonal air-quality averages (all years pooled)")

**Interpretation (Seasonal patterns):**

Seasonality is the dominant driver of month-to-month AQI variability.
**Winter** (December–February) exhibits the highest concentrations — mean AQI
exceeding 215 and mean PM2.5 above 154 µg/m³ — driven by temperature inversions,
low wind speeds, negligible rainfall, and peak brick-kiln activity.
**Pre-monsoon** (March–May) shows a moderate decline as rising temperatures and
occasional Nor'westers increase vertical mixing.
The **Monsoon** (June–September) brings the cleanest air, with mean AQI dropping
below 146 — a roughly 32 % reduction from winter — owing to heavy rainfall washout
and strong southerly winds.
**Post-monsoon** (October–November) marks the transitional deterioration as rainfall
recedes and atmospheric stability increases.

---
## 5. Trend Interpretation — Monthly Time Series

In [ ]:
df_sorted = df.sort_values("month_start").copy()
df_sorted["aqi_roll12"] = df_sorted["aqi_mean"].rolling(12, min_periods=6).mean()
df_sorted["pm25_roll12"] = df_sorted["pm25_mean"].rolling(12, min_periods=6).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), dpi=DPI, sharex=True)

# AQI panel
axes[0].plot(df_sorted["month_start"], df_sorted["aqi_mean"],
             marker=".", markersize=4, linewidth=0.8, alpha=0.55, label="Monthly mean AQI")
axes[0].plot(df_sorted["month_start"], df_sorted["aqi_roll12"],
             linewidth=2.2, color="#D65F5F", label="12-month rolling average")
axes[0].axhline(150, ls="--", color="orange", lw=1, alpha=0.7, label="AQI 150 (Unhealthy threshold)")
axes[0].set_ylabel("AQI (US EPA)", fontsize=11)
axes[0].legend(fontsize=9, loc="upper right")
axes[0].set_title("(a) Monthly AQI with 12-Month Rolling Average", fontsize=12)

# PM2.5 panel
axes[1].plot(df_sorted["month_start"], df_sorted["pm25_mean"],
             marker=".", markersize=4, linewidth=0.8, alpha=0.55, label="Monthly mean PM2.5")
axes[1].plot(df_sorted["month_start"], df_sorted["pm25_roll12"],
             linewidth=2.2, color="#D65F5F", label="12-month rolling average")
axes[1].axhline(35, ls="--", color="green", lw=1, alpha=0.7, label="WHO IT-1 (35 µg/m³ annual)")
axes[1].axhline(15, ls=":", color="darkgreen", lw=1, alpha=0.7, label="WHO Guideline (15 µg/m³ annual)")
axes[1].set_ylabel("PM2.5 (µg/m³)", fontsize=11)
axes[1].set_xlabel("Date", fontsize=11)
axes[1].legend(fontsize=9, loc="upper right")
axes[1].set_title("(b) Monthly PM2.5 with 12-Month Rolling Average", fontsize=12)

fig.suptitle("Figure 4 — Long-Term Trend in Dhaka Air Quality (2017–2025)",
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Mann-Kendall style trend using Theil-Sen slope on monthly AQI
t_numeric = np.arange(len(df_sorted))
slope_aqi, intercept_aqi, r_aqi, p_aqi, _ = stats.linregress(t_numeric, df_sorted["aqi_mean"])
slope_pm, intercept_pm, r_pm, p_pm, _ = stats.linregress(t_numeric, df_sorted["pm25_mean"])

print("Linear trend (OLS on monthly data):")
print(f"  AQI  : slope = {slope_aqi:+.3f} per month ({slope_aqi*12:+.2f}/year), "
      f"R² = {r_aqi**2:.3f}, p = {p_aqi:.4f}")
print(f"  PM2.5: slope = {slope_pm:+.3f} per month ({slope_pm*12:+.2f}/year), "
      f"R² = {r_pm**2:.3f}, p = {p_pm:.4f}")

**Interpretation (Long-term trend):**

The 12-month rolling average smooths out the strong seasonal cycle and reveals
the underlying interannual trajectory. Visual inspection and the OLS trend
statistics suggest a modest downward trend in both AQI and PM2.5 over the
nine-year window, though the magnitude is small relative to the seasonal
amplitude. The trend is consistent with incremental emission controls
(e.g., brick-kiln conversion, vehicle-emission standards) but **Dhaka
remains far above both WHO guideline levels and Bangladesh national
ambient air-quality standards throughout the entire period**.
Structural breaks — for example during COVID-19 lockdowns in 2020 — are not
strongly visible in the rolling average, suggesting that the pandemic's
air-quality effect was transient.

---
## 6. Correlation Analysis

In [ ]:
corr_vars = ["aqi_mean", "pm25_mean", "population_total",
             "urban_population", "rural_population", "hdi", "poverty_rate_pct"]
corr_labels = ["AQI (mean)", "PM2.5 (mean)", "Population",
               "Urban pop.", "Rural pop.", "HDI", "Poverty %"]

corr_matrix = df[corr_vars].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(9, 7), dpi=DPI)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Figure 5 — Correlation Matrix: Air Quality vs. Socioeconomic Indicators",
             fontsize=12, pad=14)
fig.tight_layout()
plt.show()

In [ ]:
# Correlation table
corr_display = corr_matrix.round(3)
corr_display.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).set_caption(
    "Table 4 — Pearson correlation coefficients (air quality × socioeconomic indicators)"
)

In [ ]:
# Scatter plots for key bivariate relationships
scatter_pairs = [
    ("population_total", "pm25_mean", "Population (total)", "PM2.5 (µg/m³)"),
    ("urban_population", "pm25_mean", "Urban population", "PM2.5 (µg/m³)"),
    ("hdi", "aqi_mean", "HDI", "AQI (mean)"),
    ("poverty_rate_pct", "aqi_mean", "Poverty rate (%)", "AQI (mean)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=DPI)
axes = axes.ravel()

for idx, (xvar, yvar, xlabel, ylabel) in enumerate(scatter_pairs):
    ax = axes[idx]
    ax.scatter(df[xvar], df[yvar], s=28, alpha=0.55, edgecolors="white", linewidth=0.4)
    # Regression line
    sl, ic, r, p, _ = stats.linregress(df[xvar], df[yvar])
    x_fit = np.linspace(df[xvar].min(), df[xvar].max(), 100)
    ax.plot(x_fit, sl * x_fit + ic, color="#D65F5F", linewidth=1.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f"r = {r:.2f},  p = {p:.3g}", fontsize=10)
    # Format x-axis for population
    if "population" in xvar.lower():
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e6:.0f}M"))

fig.suptitle("Figure 6 — Scatter Plots of Air Quality vs. Socioeconomic Variables",
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

**Interpretation (Correlations):**

Bivariate Pearson correlations reveal strong negative associations between
air-quality indicators and development proxies (HDI, urban population) and a
moderate positive correlation with the poverty rate. **These cross-sectional
associations should be interpreted with caution**: because all variables trend
over time (population rises monotonically, poverty falls), the correlations
largely reflect **shared temporal trends** rather than causal mechanisms.

Specifically:
- Rising population and urbanisation coincide with a period of *declining* AQI
  and PM2.5, producing **negative** r-values. This does *not* imply that
  population growth improves air quality; instead, both variables are
  confounded by time.
- The positive correlation between poverty rate and AQI is similarly time-driven:
  higher historical pollution coincides with higher historical poverty.
- Establishing causal links would require controlling for meteorological
  covariates, economic activity, and policy interventions within a multivariate
  framework.

---
## 7. Short Results Narrative

### Draft Results Section

Using monthly US Embassy monitoring data from January 2017 to December 2025,
we characterise Dhaka's ambient PM2.5 and AQI across **108 complete months**.

**Overall levels.** Monthly mean PM2.5 concentrations averaged approximately
115 µg/m³ across the study period, with the corresponding AQI averaging
around 181 — classifying Dhaka's air as *Unhealthy* on the US EPA scale for
the majority of the year. Even the cleanest months (monsoon lows) rarely
fell below the *Moderate* threshold (AQI < 100).

**Seasonality.** The seasonal cycle is the single strongest determinant of
month-to-month air-quality variation. Winter months exhibit mean PM2.5
concentrations exceeding 154 µg/m³ (AQI ≈ 215), driven by low boundary-layer
heights, negligible precipitation, and peak brick-kiln emissions. Monsoon
months bring substantial relief, with mean PM2.5 dropping to ≈ 89 µg/m³
(AQI ≈ 146), a reduction of approximately 42 % relative to winter.

**Interannual trend.** A linear regression on monthly AQI yields a modest
downward slope, but absolute levels remain far above WHO guidelines
(annual mean PM2.5 ≤ 15 µg/m³) and even the least stringent WHO interim
target (IT-1: 35 µg/m³). The 12-month rolling average shows no year in
which Dhaka approaches acceptable air quality.

**Socioeconomic correlates.** Bivariate correlations between air quality and
population, HDI, and poverty rate are statistically significant but primarily
reflect shared temporal trends. Causal attribution would require a
multivariate model controlling for meteorology and policy interventions.

**Implications.** The persistence of hazardous air quality across nearly a
decade — despite incremental regulatory efforts — underscores the need for
transformative interventions targeting the dominant source sectors
(transport, construction, brick kilns) and for continuous, quality-assured
monitoring to evaluate policy efficacy.